# Chien luoc Close > MA10 thi Buy va MA10 > MA20; 
### Close < MA10 va MA10 < 20 thi Sell
=> Làm sao để code được

In [15]:
# Lay du lieu 1m
import ccxt # ccxt hoac binance
import pandas as pd

# Khởi tạo sàn giao dịch Binance
exchange = ccxt.binance() # Hoặc có thể sử dụng ccxt.coinbase() nếu muốn kết nối với Coinbase
# exchange = ccxt.coinbase()
# exchange = ccxt.mexc()

# Đặt cặp giao dịch (BTC/USDT) và khoảng thời gian (H1 là mỗi giờ)
symbol = 'NEAR/USDT'
timeframe = '1m'

# Lấy lịch sử giá
ohlcv = exchange.fetch_ohlcv(symbol, timeframe, limit=1000) # Ham/ EndPoint fetch_ohlcv
ohlcv

[[1763674380000, 2.123, 2.126, 2.122, 2.126, 8896.0],
 [1763674440000, 2.126, 2.126, 2.121, 2.122, 8588.7],
 [1763674500000, 2.122, 2.124, 2.121, 2.122, 2416.3],
 [1763674560000, 2.122, 2.128, 2.121, 2.128, 12152.9],
 [1763674620000, 2.127, 2.131, 2.126, 2.128, 60630.6],
 [1763674680000, 2.127, 2.129, 2.125, 2.125, 13285.5],
 [1763674740000, 2.126, 2.127, 2.124, 2.126, 11927.3],
 [1763674800000, 2.126, 2.13, 2.126, 2.129, 13700.2],
 [1763674860000, 2.13, 2.131, 2.127, 2.131, 18298.9],
 [1763674920000, 2.13, 2.132, 2.128, 2.131, 6265.9],
 [1763674980000, 2.131, 2.132, 2.129, 2.132, 9756.3],
 [1763675040000, 2.133, 2.136, 2.133, 2.136, 11481.8],
 [1763675100000, 2.136, 2.136, 2.134, 2.134, 5614.1],
 [1763675160000, 2.134, 2.136, 2.133, 2.135, 7706.6],
 [1763675220000, 2.135, 2.139, 2.134, 2.139, 11638.6],
 [1763675280000, 2.138, 2.141, 2.138, 2.14, 23200.2],
 [1763675340000, 2.139, 2.141, 2.137, 2.138, 17604.5],
 [1763675400000, 2.138, 2.138, 2.132, 2.134, 33552.6],
 [1763675460000, 2.13

In [16]:
# Parse data
import pandas as pd

data = pd.DataFrame(ohlcv)
data.columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
data['timestamp'] = pd.to_datetime(data['timestamp'], unit='ms')
data.set_index('timestamp', inplace=True)
data.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'}, inplace=True)
data

,Open,High,Low,Close,Volume
timestamp,,,,,
2025-11-20 21:33:00,2.123,2.126,2.122,2.126,8896.0
2025-11-20 21:34:00,2.126,2.126,2.121,2.122,8588.7
2025-11-20 21:35:00,2.122,2.124,2.121,2.122,2416.3
2025-11-20 21:36:00,2.122,2.128,2.121,2.128,12152.9
2025-11-20 21:37:00,2.127,2.131,2.126,2.128,60630.6
...,...,...,...,...,...
2025-11-21 14:08:00,1.922,1.926,1.922,1.926,17966.4
2025-11-21 14:09:00,1.927,1.929,1.927,1.928,12849.0
2025-11-21 14:10:00,1.928,1.928,1.922,1.922,10478.0


# Process

In [17]:
# Process
import talib
data['MA10'] = talib.SMA(data['Close'], timeperiod=10)
data['MA20'] = talib.SMA(data['Close'], timeperiod=20)

# Chien luoc Close > MA10 va MA10 > MA20 thi Buy_Signal
data['Buy_Signal'] = (data['Close'] > data['MA10']) & (data['MA10'] > data['MA20'])
# data['Buy_Signal'] = False

# Chien luoc Close < MA10 va MA10 < MA20 thi Sell_Signal
data['Sell_Signal'] = (data['Close'] < data['MA10']) & (data['MA10'] < data['MA20'])
# data['Sell_Signal'] = True

data


,Open,High,Low,Close,Volume,MA10,MA20,Buy_Signal,Sell_Signal
timestamp,,,,,,,,,
2025-11-20 21:33:00,2.123,2.126,2.122,2.126,8896.0,NaN,NaN,False,False
2025-11-20 21:34:00,2.126,2.126,2.121,2.122,8588.7,NaN,NaN,False,False
2025-11-20 21:35:00,2.122,2.124,2.121,2.122,2416.3,NaN,NaN,False,False
2025-11-20 21:36:00,2.122,2.128,2.121,2.128,12152.9,NaN,NaN,False,False
2025-11-20 21:37:00,2.127,2.131,2.126,2.128,60630.6,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...
2025-11-21 14:08:00,1.922,1.926,1.922,1.926,17966.4,1.9252,1.92995,False,False
2025-11-21 14:09:00,1.927,1.929,1.927,1.928,12849.0,1.9251,1.92960,False,False
2025-11-21 14:10:00,1.928,1.928,1.922,1.922,10478.0,1.9245,1.92890,False,True


In [18]:
# last_record = data.iloc[-2]
# last_record

In [19]:
# Day sang redis
import redis
import numpy as np
from datetime import datetime
# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
# Nếu có tín hiệu thì đẩy qua Redis
# Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line     Buy_Signal      Sell_Signal
# Tạo kết nối Redis
r = redis.Redis(host='localhost', port=6379, db=10) # 0-5 CK, 6-10 Crypto, 11-15 Forex
# Tạo hash key
hash_key = 'Buoi 6_OG'
last_record = data.iloc[-2]

# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
    for field, value in last_record.to_dict().items():
        # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
        if isinstance(value, pd.Timestamp):
            value = value.isoformat()
        elif isinstance(value, (int, np.uint64)):
            value = str(value)
        r.hset(hash_key, field, value)  
        r.hset(hash_key, 'Symbol', symbol)
        r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
        last_record['Insertdate'] = datetime.now().isoformat()
    print(last_record)   
else:
    print(last_record)   
    print('Không có tín hiệu!')

Open                                1.922
High                                1.923
Low                                 1.918
Close                               1.919
Volume                            15313.7
MA10                                1.924
MA20                              1.92775
Buy_Signal                          False
Sell_Signal                          True
Insertdate     2025-11-21T21:12:16.370694
Name: 2025-11-21 14:11:00, dtype: object


C:\Users\PCDTT\AppData\Local\Temp\ipykernel_11152\1403567112.py:25: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\PCDTT\AppData\Local\Temp\ipykernel_11152\1403567112.py:25: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



# Code them bieu do

In [23]:
# Ve bieu do cua Data o tren: Ve duong Close, MA10, MA20, tin hieu Buy_Signal la triangle blue, tin hieu Sell_Signal la triangle red
import plotly.graph_objects as go

# Tạo figure
fig = go.Figure()

# Vẽ Candlestick (nến)
fig.add_trace(go.Candlestick(
    x=data.index,
    open=data['Open'],
    high=data['High'],
    low=data['Low'],
    close=data['Close'],
    name='Candlestick',
    increasing_line_color='green',
    decreasing_line_color='red',
    increasing_fillcolor='green',
    decreasing_fillcolor='red'
))

# Vẽ đường MA10
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['MA10'],
    mode='lines',
    name='MA10',
    line=dict(color='blue', width=2),
    hovertemplate='<b>MA10</b><br>Time: %{x}<br>Price: %{y:.4f} USDT<extra></extra>'
))

# Vẽ đường MA20
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['MA20'],
    mode='lines',
    name='MA20',
    line=dict(color='orange', width=2),
    hovertemplate='<b>MA20</b><br>Time: %{x}<br>Price: %{y:.4f} USDT<extra></extra>'
))

# Vẽ Buy_Signal (triangle blue) - đặt ở Low của nến
buy_signals = data[data['Buy_Signal'] == True]
if len(buy_signals) > 0:
    fig.add_trace(go.Scatter(
        x=buy_signals.index,
        y=buy_signals['Low'],  # Đặt ở Low của nến
        mode='markers',
        name='Buy Signal',
        marker=dict(
            symbol='triangle-up',
            size=15,
            color='blue',
            line=dict(width=1, color='darkblue')
        ),
        hovertemplate='<b>Buy Signal</b><br>Time: %{x}<br>Price: %{y:.4f} USDT<extra></extra>'
    ))

# Vẽ Sell_Signal (triangle red) - đặt ở High của nến
sell_signals = data[data['Sell_Signal'] == True]
if len(sell_signals) > 0:
    fig.add_trace(go.Scatter(
        x=sell_signals.index,
        y=sell_signals['High'],  # Đặt ở High của nến
        mode='markers',
        name='Sell Signal',
        marker=dict(
            symbol='triangle-down',
            size=15,
            color='red',
            line=dict(width=1, color='darkred')
        ),
        hovertemplate='<b>Sell Signal</b><br>Time: %{x}<br>Price: %{y:.4f} USDT<extra></extra>'
    ))

# Thiết lập layout
fig.update_layout(
    title=dict(
        text=f'{symbol} - Candlestick Chart with MA10, MA20 and Trading Signals',
        font=dict(size=16, color='black')
    ),
    xaxis=dict(
        title='Time',
        showgrid=True,
        gridcolor='lightgray',
        rangeslider=dict(visible=False),  # Ẩn range slider mặc định
        type='date'
    ),
    yaxis=dict(
        title='Price (USDT)',
        showgrid=True,
        gridcolor='lightgray'
    ),
    hovermode='x unified',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    height=600,
    template='plotly_white'
)

# Hiển thị biểu đồ
fig.show()
